In [55]:
import pandas as pd
import numpy as np
import re
import os
import joblib
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_absolute_error, r2_score
from datetime import datetime
import glob
import logging

# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("model_artifacts/pipeline_log.log"),
        logging.StreamHandler()
    ]
)

print(f"LightGBM version: {lgb.__version__}")

# === Transformer personnalisé pour preprocessing ===
class DataFramePreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.cat_cols = ['marque', 'modele', 'carburant', 'boite_de_vitesse', 'couleur', 'ville']
        self.scaler = StandardScaler()
        self.top_categories = {}
        
    def fit(self, X, y=None):
        df = self._preprocess(X.copy())
        for col in self.cat_cols:
            self.top_categories[col] = df[col].value_counts().nlargest(50).index.tolist()
        df_enc = self._encode_and_scale(df)
        return self

    def transform(self, X):
        df = self._preprocess(X.copy())
        df_enc = self._encode_and_scale(df)
        return df_enc

    def _clean_price(self, x):
        if pd.isna(x):
            return np.nan
        cleaned = re.sub(r"[^\d]", "", str(x))
        return float(cleaned) if cleaned else np.nan

    def _clean_text(self, text):
        if pd.isna(text):
            return text
        text = re.sub(r'[^\w\s]', '_', text)
        text = re.sub(r'\s+', '_', text)
        return text
    
    def _preprocess(self, df):
        logging.info("Début du préprocessing des données")
        df['prix'] = df['prix'].apply(self._clean_price)
        df['puissance_fiscale'] = df['puissance_fiscale'].astype(str).str.replace(" CV", "", regex=False)
        df['puissance_fiscale'] = pd.to_numeric(df['puissance_fiscale'], errors='coerce')
        df['mise_en_circulation'] = pd.to_numeric(df['mise_en_circulation'], errors='coerce')
        df['kilometrage'] = pd.to_numeric(df['kilometrage'], errors='coerce')
        df['date_scraping'] = pd.to_datetime(df['date_scraping'], errors='coerce')
        df['date_annonce'] = pd.to_datetime(df['date_annonce'], errors='coerce')

        filt = ~df['titre'].str.contains(
            r'airbag|pi[eè]ce|accessoire|jante|volant|pneu|pare[-\s]?choc|feu|phare|capot|moteur|boîte',
            case=False, na=False)
        df = df[filt]
        logging.info(f"Lignes restantes après filtrage du titre: {df.shape[0]}")

        df = df[
            df['prix'].notnull() &
            df['kilometrage'].notnull() &
            df['puissance_fiscale'].notnull() &
            df['marque'].notnull() &
            df['modele'].notnull() &
            df['carburant'].notnull() &
            df['boite_de_vitesse'].notnull()
        ]
        logging.info(f"Lignes restantes après filtrage des colonnes manquantes: {df.shape[0]}")

        for col in ['prix', 'kilometrage', 'puissance_fiscale']:
            if df.empty:
                logging.warning(f"DataFrame vide avant filtrage des outliers pour {col}")
                break
            q_low = df[col].quantile(0.05)
            q_high = df[col].quantile(0.95)
            logging.info(f"Seuils pour {col}: min={q_low:.2f}, max={q_high:.2f}")
            df = df[(df[col] >= q_low) & (df[col] <= q_high)]
            logging.info(f"Lignes restantes après filtrage des outliers pour {col}: {df.shape[0]}")

        if not df.empty:
            for col in ['prix', 'kilometrage', 'puissance_fiscale']:
                df[col] = np.log1p(df[col])

            for col in self.cat_cols:
                df[col] = df[col].apply(self._clean_text)

            df['age_vehicule'] = 2025 - df['mise_en_circulation']
            df['age_vehicule'] = df['age_vehicule'].apply(lambda x: x if x >= 0 else 0)
            df['km_par_an'] = df.apply(lambda r: r['kilometrage'] / r['age_vehicule'] if r['age_vehicule'] > 0 else r['kilometrage'], axis=1)
            df['mois_annonce'] = df['date_annonce'].dt.month.fillna(0).astype(int)
            df['annee_annonce'] = df['date_annonce'].dt.year.fillna(0).astype(int)
            df['has_premium_features'] = df['description'].str.contains(
                r'cuir|toit_ouvrant|gps|clim|camera|xenon|led|navigatio|toit_panoramique|sieges_chauffants',
                case=False, na=False).astype(int)
            df['desc_length'] = df['description'].str.len().fillna(0)
            df['price_per_km'] = df['prix'] / df['kilometrage']
            df['is_high_price'] = (df['prix'] > df['prix'].quantile(0.75)).astype(int)
            df['power_age_interaction'] = df['puissance_fiscale'] * df['age_vehicule']

            to_drop = ['id', 'url_annonce', 'description', 'Numero_telephone', 'titre', 'date_scraping', 'date_annonce',
                       'source', 'etat', 'vendeur']  # Ajout des colonnes problématiques
            for col in to_drop:
                if col in df.columns:
                    df.drop(columns=col, inplace=True)

            df.drop_duplicates(inplace=True)

        logging.info(f"Preprocessing terminé, shape finale: {df.shape}")
        if df.shape[0] == 0:
            logging.warning("Avertissement : Aucune ligne restante après le prétraitement. Vérifiez les données d'entrée.")
        return df

    def _encode_and_scale(self, df):
        if df.empty:
            logging.warning("DataFrame vide après prétraitement. Retour d'un DataFrame vide.")
            return df
        for col in self.cat_cols:
            top_n = self.top_categories.get(col, [])
            df[col] = df[col].apply(lambda x: x if x in top_n else 'Other')

        df = pd.get_dummies(df, columns=self.cat_cols, drop_first=True)
        df.columns = [re.sub(r'[^\w]', '_', col) for col in df.columns]

        num_cols = ['age_vehicule', 'km_par_an', 'has_premium_features', 'power_age_interaction', 'desc_length', 'price_per_km', 'is_high_price']
        num_cols = [c for c in num_cols if c in df.columns]
        if num_cols:
            df[num_cols] = self.scaler.fit_transform(df[num_cols])
        return df

# === Chargement automatisé de plusieurs fichiers CSV ===
def load_csv_files(input_dir):
    logging.info(f"Chargement des fichiers CSV depuis: {input_dir}")
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        logging.error("Aucun fichier CSV trouvé dans le répertoire")
        raise FileNotFoundError("Aucun fichier CSV trouvé dans le répertoire")
    
    dfs = []
    for file in csv_files:
        logging.info(f"Chargement du fichier: {file}")
        df = pd.read_csv(file, sep=';', encoding='utf-8')
        dfs.append(df)
    
    combined_df = pd.concat(dfs, ignore_index=True)
    logging.info(f"Shape combinée des données: {combined_df.shape}")
    return combined_df

# === Pipeline complet avec automatisation ===
def automated_pipeline(input_dir, output_dir="model_artifacts"):
    os.makedirs(output_dir, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # Chargement des données
    df = load_csv_files(input_dir)
    
    preprocessor = DataFramePreprocessor()
    X = df.drop(columns=['prix'], errors='ignore')
    y = df['prix'] if 'prix' in df.columns else None
    
    X_processed = preprocessor.fit_transform(df)
    y = X_processed['prix']
    X_processed = X_processed.drop(columns=['prix'])
    X_processed = X_processed.select_dtypes(include=['int64', 'float64', 'bool'])

    y_quartiles = pd.qcut(y, 4, labels=False)
    X_train, X_test, y_train, y_test = train_test_split(
        X_processed, y, test_size=0.2, random_state=42, stratify=y_quartiles)

    logging.info(f"Train: {X_train.shape}, Test: {X_test.shape}")

    X_train.columns = [re.sub(r'[^\w]', '_', col) for col in X_train.columns]
    X_test.columns = [re.sub(r'[^\w]', '_', col) for col in X_test.columns]
    X_train = X_train.loc[:, ~X_train.columns.duplicated()]
    X_test = X_test.loc[:, ~X_test.columns.duplicated()]
    
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'verbosity': -1,
        'boosting_type': 'gbdt',
        'seed': 42,
        'learning_rate': 0.005,
        'num_leaves': 40,
        'max_depth': 10,
        'min_data_in_leaf': 70,
        'feature_fraction': 0.7,
        'bagging_fraction': 0.7,
        'bagging_freq': 5,
        'lambda_l1': 0.7,
        'lambda_l2': 0.7,
        'min_child_weight': 2,
        'is_unbalance': True
    }

    cat_cols = preprocessor.cat_cols
    categorical_features = [col for col in X_train.columns if any(col.startswith(prefix) for prefix in cat_cols)]
    train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=categorical_features)
    valid_data = lgb.Dataset(X_test, label=y_test, categorical_feature=categorical_features)

    evals_result = {}
    model = lgb.train(
        params,
        train_data,
        early_stopping_rounds=100,
        num_boost_round=5000,
        valid_sets=[train_data, valid_data],
        valid_names=['train', 'valid'],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=False),
            lgb.log_evaluation(period=10),
            lgb.record_evaluation(evals_result)
        ]
    )

    y_pred = model.predict(X_test)
    y_pred_orig = np.expm1(y_pred)
    y_test_orig = np.expm1(y_test)
    rmse = np.sqrt(np.mean((y_pred_orig - y_test_orig) ** 2))
    r2 = r2_score(y_test_orig, y_pred_orig)
    mae = mean_absolute_error(y_test_orig, y_pred_orig)

    logging.info(f"Final RMSE: {rmse:.2f}")
    logging.info(f"Final R²: {r2:.4f}")
    logging.info(f"Final MAE: {mae:.2f}")

    plt.figure(figsize=(10, 6))
    plt.plot(evals_result['train']['rmse'], label='Train RMSE', color='blue')
    plt.plot(evals_result['valid']['rmse'], label='Validation RMSE', color='orange')
    plt.xlabel('Boosting Rounds')
    plt.ylabel('RMSE')
    plt.title('Training and Validation Loss Curves')
    plt.legend()
    plt.grid(True)
    plt.savefig(os.path.join(output_dir, f"loss_curve_{timestamp}.png"))
    plt.close()

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('model', lgb.LGBMRegressor(**params, n_estimators=model.current_iteration()))
    ])
    pipeline.named_steps['model'].fit(X_processed, y)

    model_path = os.path.join(output_dir, f"lgbm_pipeline_model_{timestamp}.pkl")
    joblib.dump(pipeline, model_path)
    logging.info(f"Pipeline complet sauvegardé dans: {model_path}")

    return pipeline, model

LightGBM version: 4.6.0


In [56]:
import pandas as pd
import numpy as np
import glob
import os
import joblib
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from datetime import datetime

# Configuration
input_dir = "C:/Users/maram/Downloads/annonces_brutes_202507021126/"
model_path = "model_artifacts/lgbm_pipeline_model_20250709_111048.pkl"  # Mettre à jour avec le dernier modèle

# Test 0 : Examiner les données brutes (pour débogage)
print("Test 0 : Examen des données brutes")
try:
    df = load_csv_files(input_dir)
    print("Aperçu des 10 premières lignes brutes :\n", df.head(10))
    print("Valeurs manquantes dans les 10 premières lignes :\n", df.head(10).isnull().sum())
    print("Colonnes de type object dans les 10 premières lignes :\n", 
          df.head(10).select_dtypes(include=['object']).columns)
    print("Contenu des colonnes source, etat, vendeur :\n", 
          df.head(10)[['source', 'etat', 'vendeur']])
except Exception as e:
    logging.error("Erreur lors du chargement des données brutes : %s", str(e))
    print("Erreur lors du chargement des données brutes :", e)

# Test 1 : Vérifier le chargement des données
print("\nTest 1 : Chargement des fichiers CSV")
try:
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    if not csv_files:
        print("Erreur : Aucun fichier CSV trouvé dans", input_dir)
    else:
        print("Fichiers CSV trouvés :", csv_files)
        print("Shape des données chargées :", df.shape)
        print("Colonnes :", df.columns)
except Exception as e:
    logging.error("Erreur lors du chargement des données : %s", str(e))
    print("Erreur lors du chargement des données :", e)

# Test 2 : Vérifier le prétraitement
print("\nTest 2 : Prétraitement des données")
try:
    preprocessor = DataFramePreprocessor()
    X_processed = preprocessor.fit_transform(df)
    print("Shape après prétraitement :", X_processed.shape)
    print("Colonnes après prétraitement :", X_processed.columns)
    print("Aperçu des données prétraitées :\n", X_processed.head())
except Exception as e:
    logging.error("Erreur lors du prétraitement : %s", str(e))
    print("Erreur lors du prétraitement :", e)

# Test 3 : Exécuter le pipeline complet
print("\nTest 3 : Exécution du pipeline complet")
try:
    pipeline, model = automated_pipeline(input_dir)
    print("Pipeline exécuté avec succès. Vérifiez model_artifacts pour les résultats.")
except Exception as e:
    logging.error("Erreur lors de l'exécution du pipeline : %s", str(e))
    print("Erreur lors de l'exécution du pipeline :", e)

# Test 4 : Tester les prédictions
print("\nTest 4 : Prédictions avec le modèle sauvegardé")
try:
    pipeline = joblib.load(model_path)
    # Sélectionner un échantillon valide (lignes qui passent les filtres)
    preprocessor = DataFramePreprocessor()
    df_processed = preprocessor.fit_transform(df)
    if df_processed.empty:
        logging.warning("Erreur : Aucun échantillon valide après prétraitement. Vérifiez les données d'entrée.")
        print("Erreur : Aucun échantillon valide après prétraitement. Vérifiez les données d'entrée.")
    else:
        # Sélectionner un échantillon de 50 lignes
        df_sample = df_processed.head(50)
        if df_sample.empty:
            logging.warning("Erreur : L'échantillon de 50 lignes est vide après prétraitement.")
            print("Erreur : L'échantillon de 50 lignes est vide après prétraitement.")
        else:
            # S'assurer que seules les colonnes numériques sont utilisées
            df_sample_numeric = df_sample.select_dtypes(include=['int64', 'float64', 'bool']).drop(columns=['prix'])
            if df_sample_numeric.empty:
                logging.warning("Erreur : Aucune colonne numérique valide dans l'échantillon.")
                print("Erreur : Aucune colonne numérique valide dans l'échantillon.")
            else:
                predictions = pipeline.named_steps['model'].predict(df_sample_numeric)
                predictions_orig = np.expm1(predictions)
                logging.info("Prédictions (échelle réelle) : %s", predictions_orig)
                print("Prédictions (échelle réelle) :", predictions_orig)
                if 'prix' in df_sample.columns:
                    y_true = np.expm1(df_sample['prix'])
                    logging.info("Prix réels : %s", y_true.values)
                    print("Prix réels :", y_true.values)
                    differences = abs(predictions_orig - y_true.values)
                    logging.info("Différences absolues : %s", differences)
                    print("Différences absolues :", differences)
                    # Calculer et afficher R², RMSE, et MAE
                    r2 = r2_score(y_true, predictions_orig)
                    rmse = np.sqrt(mean_squared_error(y_true, predictions_orig))
                    mae = mean_absolute_error(y_true, predictions_orig)
                    logging.info(f"Final R²: {r2:.4f}")
                    logging.info(f"Final RMSE: {rmse:.2f}")
                    logging.info(f"Final MAE: {mae:.2f}")
                    print(f"Final R²: {r2:.4f}")
                    print(f"Final RMSE: {rmse:.2f}")
                    print(f"Final MAE: {mae:.2f}")
                    # Créer un graphique de dispersion
                    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                    plt.figure(figsize=(10, 6))
                    plt.scatter(y_true, predictions_orig, color='blue', alpha=0.5, label='Prédictions')
                    plt.plot([y_true.min(), y_true.max()], [y_true.min(), y_true.max()], 'r--', label='Ligne idéale')
                    plt.xlabel('Prix réels (€)')
                    plt.ylabel('Prix prédits (€)')
                    plt.title('Prédictions vs Prix réels (Échantillon de 50 lignes)')
                    plt.legend()
                    plt.grid(True)
                    plt.savefig(os.path.join("model_artifacts", f"predictions_scatter_{timestamp}.png"))
                    plt.close()
                    logging.info(f"Graphique de dispersion sauvegardé : model_artifacts/predictions_scatter_{timestamp}.png")
except Exception as e:
    logging.error("Erreur lors des prédictions : %s", str(e))
    print("Erreur lors des prédictions :", e)

# Test 5 : Vérifier le fichier de log
print("\nTest 5 : Vérification du fichier de log")
log_path = "model_artifacts/pipeline_log.log"
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        print("Dernières lignes du log :\n", ''.join(f.readlines()[-10:]))
else:
    print("Erreur : Fichier de log non trouvé.")

2025-07-09 15:02:43,558 - INFO - Chargement des fichiers CSV depuis: C:/Users/maram/Downloads/annonces_brutes_202507021126/
2025-07-09 15:02:43,563 - INFO - Chargement du fichier: C:/Users/maram/Downloads/annonces_brutes_202507021126\annonces_brutes_202507021126.csv


Test 0 : Examen des données brutes


C:\Users\maram\AppData\Local\Temp\ipykernel_20100\256096098.py:160: DtypeWarning: Columns (21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, sep=';', encoding='utf-8')
2025-07-09 15:02:47,847 - INFO - Shape combinée des données: (356042, 22)
2025-07-09 15:02:48,060 - INFO - Début du préprocessing des données


Aperçu des 10 premières lignes brutes :
                          id  \
0  00006b3949814c0bba59402f   
1  00013d503b984ade93365bf4   
2  00033595bf784a4a8d78eefa   
3  0003c747fcd9487687dd8928   
4  00060c3631184a8eb9d19b04   
5  00066bec96284b4e9b6d78bd   
6  00069110f5674770abc1db53   
7  0006a925eb444ae29df9a9f0   
8  0006c86020f34587884bca65   
9  0006cb57bcd149878529bbf8   

                                               titre  \
0            🚗 À VENDRE : Volkswagen Polo 7 – 2011 🚗   
1                          Très belle Renault captur   
2        SUZUKI DZIRE PREMIÈRE MAIN EN TRÈS BON ÉTAT   
3                                        Peugeot 206   
4                         Airbag volant opel Astra H   
5  Skoda Octavia 1.5 Tsi dsg 7 2017 klm 157.000 E...   
6                  MG zs première main toute options   
7                  BMW X5 Diesel 3L échange possible   
8                                     a ne pas rater   
9                                  Renault clio 2021   



2025-07-09 15:02:51,503 - INFO - Lignes restantes après filtrage du titre: 345726
2025-07-09 15:02:51,663 - INFO - Lignes restantes après filtrage des colonnes manquantes: 238051
2025-07-09 15:02:51,673 - INFO - Seuils pour prix: min=0.00, max=95000.00
2025-07-09 15:02:51,775 - INFO - Lignes restantes après filtrage des outliers pour prix: 226254
2025-07-09 15:02:51,790 - INFO - Seuils pour kilometrage: min=1.00, max=350000.00
2025-07-09 15:02:51,922 - INFO - Lignes restantes après filtrage des outliers pour kilometrage: 204819
2025-07-09 15:02:51,933 - INFO - Seuils pour puissance_fiscale: min=4.00, max=13.00
2025-07-09 15:02:52,024 - INFO - Lignes restantes après filtrage des outliers pour puissance_fiscale: 190196
2025-07-09 15:03:01,184 - INFO - Preprocessing terminé, shape finale: (182328, 21)
2025-07-09 15:03:02,148 - INFO - Début du préprocessing des données
2025-07-09 15:03:05,990 - INFO - Lignes restantes après filtrage du titre: 345726
2025-07-09 15:03:06,176 - INFO - Lignes 

...

Erreur lors de l'exécution du pipeline : train() got an unexpected keyword argument 'early_stopping_rounds'

Test 4 : Prédictions avec le modèle sauvegardé


2025-07-09 15:03:55,857 - INFO - Début du préprocessing des données
2025-07-09 15:04:00,276 - INFO - Lignes restantes après filtrage du titre: 345726
2025-07-09 15:04:00,447 - INFO - Lignes restantes après filtrage des colonnes manquantes: 238051
2025-07-09 15:04:00,456 - INFO - Seuils pour prix: min=0.00, max=95000.00
2025-07-09 15:04:00,567 - INFO - Lignes restantes après filtrage des outliers pour prix: 226254
2025-07-09 15:04:00,582 - INFO - Seuils pour kilometrage: min=1.00, max=350000.00
2025-07-09 15:04:00,716 - INFO - Lignes restantes après filtrage des outliers pour kilometrage: 204819
2025-07-09 15:04:00,730 - INFO - Seuils pour puissance_fiscale: min=4.00, max=13.00
2025-07-09 15:04:00,852 - INFO - Lignes restantes après filtrage des outliers pour puissance_fiscale: 190196
2025-07-09 15:04:11,119 - INFO - Preprocessing terminé, shape finale: (182328, 21)
2025-07-09 15:04:12,058 - INFO - Début du préprocessing des données
2025-07-09 15:04:16,852 - INFO - Lignes restantes aprè

Prédictions (échelle réelle) : [4.03917824e+04 5.16523325e+04 3.41081087e+01 3.52607834e+04
 4.10706307e+01 6.53531388e+04 9.28574498e-01 5.83472832e+01
 2.71207056e+04 1.92066786e+04 2.03479047e+04 9.76750539e-01
 8.98617151e+04 7.60289440e+04 2.73782379e+04 2.32141117e+04
 2.36530339e+04 2.15095371e+04 6.46658095e+04 4.62539750e+04
 3.92269353e+01 1.81615998e+04 3.71087127e+04 3.75727924e+04
 2.23637662e+01 4.01807730e+02 1.17813174e+04 6.39582438e+04
 2.42982686e+04 3.92885841e+04 8.98349656e+04 2.17140661e+04
 2.52042778e+04 2.96974376e+02 5.58719856e+04 9.76750539e-01
 3.31321958e+04 2.39290102e+04 1.60225258e+04 5.22099865e+04
 1.65610576e+04 2.45731342e+04 4.82395646e+04 3.33642583e+04
 6.25740779e+04 2.31739815e+04 2.95183660e+00 1.71497407e+03
 1.79999032e+02 8.57277453e+04]
Prix réels : [3.7500e+04 4.8500e+04 3.3000e+01 3.6000e+04 3.6000e+01 6.3000e+04
 1.0000e+00 5.2000e+01 3.0990e+04 2.2000e+04 1.9700e+04 1.0000e+00
 9.0000e+04 7.5000e+04 2.5000e+04 2.1500e+04 2.4800e+04 2.

2025-07-09 15:04:29,217 - INFO - Graphique de dispersion sauvegardé : model_artifacts/predictions_scatter_20250709_150428.png



Test 5 : Vérification du fichier de log
Dernières lignes du log :
  7.01731409e+02 7.88584140e+02 1.65034403e+02 2.85933938e+02
 2.29572224e+03 2.10256242e+01 3.12801437e+03 2.32494609e-02
 1.32195759e+02 7.09898071e+01 5.22525838e+02 2.21198645e+03
 3.76105762e+03 5.73134241e+02 3.60435403e+02 1.63574173e+03
 3.25922108e+02 8.26018541e+02 4.81633955e-02 2.14974069e+02
 4.20009684e+01 7.27745314e+02]
2025-07-09 15:04:28,972 - INFO - Final R²: 0.9940
2025-07-09 15:04:28,973 - INFO - Final RMSE: 1990.97
2025-07-09 15:04:28,975 - INFO - Final MAE: 1197.88
2025-07-09 15:04:29,217 - INFO - Graphique de dispersion sauvegardé : model_artifacts/predictions_scatter_20250709_150428.png

